In [ ]:
import os
import json
import tarfile
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# 1. Train Model
data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 2. Serialize to model.json
forest_data = {
    "n_features": model.n_features_in_,
    "n_classes": getattr(model, "n_classes_", 1),
    "estimators": []
}

for est in model.estimators_:
    tree = est.tree_
    forest_data["estimators"].append({
        "children_left": tree.children_left.tolist(),
        "children_right": tree.children_right.tolist(),
        "feature": tree.feature.tolist(),
        "threshold": tree.threshold.tolist(),
        "value": tree.value.tolist()
    })

with open("model.json", "w") as f:
    json.dump(forest_data, f)

# 3. Create inference.py
os.makedirs("code", exist_ok=True)
inference_code = '''import os
import json
import numpy as np

class LightweightForest:
    def __init__(self, data):
        self.estimators = data["estimators"]

    def _predict_tree(self, tree, x):
        node = 0
        while tree["children_left"][node] != -1:
            if x[tree["feature"][node]] <= tree["threshold"][node]:
                node = tree["children_left"][node]
            else:
                node = tree["children_right"][node]
        return tree["value"][node][0][0]

    def predict(self, X):
        X = np.asarray(X)
        if X.ndim == 1:
            X = X.reshape(1, -1)
        preds = []
        for x in X:
            tree_preds = [self._predict_tree(tree, x) for tree in self.estimators]
            preds.append(np.mean(tree_preds))
        return np.array(preds)

def model_fn(model_dir):
    json_path = os.path.join(model_dir, "model.json")
    if not os.path.exists(json_path):
        for f in os.listdir(model_dir):
            if f.endswith(".json"):
                json_path = os.path.join(model_dir, f)
                break
    with open(json_path, "r") as f:
        data = json.load(f)
    return LightweightForest(data)

def input_fn(request_body, request_content_type):
    if request_content_type == "application/json":
        data = json.loads(request_body)
        if isinstance(data, dict) and "inputs" in data:
            return np.array(data["inputs"])
        return np.array(data)
    raise ValueError(f"Unsupported content type: {request_content_type}")

def predict_fn(input_data, model):
    return model.predict(input_data)

def output_fn(prediction, accept):
    if accept == "application/json":
        return json.dumps(prediction.tolist()), accept
    raise ValueError(f"Unsupported accept type: {accept}")
'''

with open("code/inference.py", "w") as f:
    f.write(inference_code)

with open("inference.py", "w") as f:
    f.write(inference_code)

# 4. Package into model.tar.gz
with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("model.json", arcname="model.json")
    tar.add("inference.py", arcname="inference.py")
    tar.add("code/inference.py", arcname="code/inference.py")

print("Created clean model.tar.gz.")

Created clean model.tar.gz.


In [ ]:
!pip install boto3 -q
import os
import boto3
import time

# Set environment variables for AWS SDK
os.environ['AWS_ACCESS_KEY_ID'] = '[YOUR_ACCESS_KEY_ID]'
os.environ['AWS_SECRET_ACCESS_KEY'] = '[YOUR_SECRET_ACCESS_KEY]'
os.environ['AWS_DEFAULT_REGION'] = 'ap-south-1'

s3 = boto3.client("s3", region_name="ap-south-1")
sagemaker = boto3.client("sagemaker", region_name="ap-south-1")

bucket_name = "rf-housing-model1"
role_arn = "arn:aws:iam::767261813063:role/SageMakerExecutionRoleForHousePrice"
image_uri = "720646828776.dkr.ecr.ap-south-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3"

model_name = "rf-housing-model-resolved"
config_name = "rf-serverless-config-resolved"
endpoint_name = "rf-housing-endpoint"

# 1. Upload artifact
print("Uploading model.tar.gz to S3...")
s3.upload_file("model.tar.gz", bucket_name, "model.tar.gz")

# 2. Cleanup stale resources if present
print("Cleaning up old resources if any exist...")
for fn, param in [
    (sagemaker.delete_endpoint, {"EndpointName": endpoint_name}),
    (sagemaker.delete_endpoint_config, {"EndpointConfigName": config_name}),
    (sagemaker.delete_model, {"ModelName": model_name})
]:
    try:
        fn(**param)
    except Exception:
        pass

time.sleep(5)

# 3. Create Model
print(f"Creating Model: {model_name}...")
sagemaker.create_model(
    ModelName=model_name,
    PrimaryContainer={
        "Image": image_uri,
        "ModelDataUrl": f"s3://{bucket_name}/model.tar.gz",
        "Environment": {
            "SAGEMAKER_PROGRAM": "inference.py",
            "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model",
            "PYTHONPATH": "/opt/ml/model:/opt/ml/model/code"
        }
    },
    ExecutionRoleArn=role_arn
)

# 4. Create Serverless Endpoint Configuration
print("Creating Endpoint Configuration...")
sagemaker.create_endpoint_config(
    EndpointConfigName=config_name,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": model_name,
        "ServerlessConfig": {
            "MemorySizeInMB": 2048,
            "MaxConcurrency": 2
        }
    }]
)

# 5. Deploy Endpoint
print("Deploying Serverless Endpoint...")
sagemaker.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=config_name
)

# 6. Monitor Status
print("Waiting for endpoint to become active...")
while True:
    desc = sagemaker.describe_endpoint(EndpointName=endpoint_name)
    status = desc["EndpointStatus"]
    print("Current Status:", status)
    if status == "InService":
        print("\nEndpoint is live and ready for demo!")
        break
    elif status == "Failed":
        print("\nFailed reason:", desc.get("FailureReason"))
        break
    time.sleep(15)

Uploading model.tar.gz to S3...
Cleaning up old resources if any exist...
Creating Model: rf-housing-model-resolved...
Creating Endpoint Configuration...
Deploying Serverless Endpoint...
Waiting for endpoint to become active...
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: Creating
Current Status: InService

Endpoint is live and ready for demo!


In [5]:
import requests

api_url = "https://on6l8u1qcb.execute-api.ap-south-1.amazonaws.com/prod"

payload = {
    "inputs": [
        [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5]
    ]
}

response = requests.post(api_url, json=payload)

if response.status_code == 200:
    data = response.json()
    raw_pred = data.get("predicted_price", [0])
    price_val = raw_pred[0] if isinstance(raw_pred, list) else raw_pred
    estimated_price = price_val * 100000 if price_val < 50 else price_val

    print("=" * 40)
    print("      HOUSE PRICE PREDICTION")
    print("=" * 40)
    print(f"Status           : SUCCESS (HTTP {response.status_code})")
    print(f"Raw Model Value  : {price_val:.4f}")
    print(f"Estimated Price  : ${estimated_price:,.2f}")
    print(f"Notification     : {data.get('notification')}")
    print("=" * 40)
else:
    print(f"Error {response.status_code}:", response.text)

      HOUSE PRICE PREDICTION
Status           : SUCCESS (HTTP 200)
Raw Model Value  : 1.5354
Estimated Price  : $153,535.04
Notification     : Email notification dispatched successfully


In [6]:
import boto3

sm = boto3.client("sagemaker", region_name="ap-south-1")

sm.delete_endpoint(EndpointName="rf-housing-endpoint")
sm.delete_endpoint_config(EndpointConfigName="rf-serverless-config-resolved")
sm.delete_model(ModelName="rf-housing-model-resolved")

print("SageMaker endpoint deleted successfully to prevent charges.")

SageMaker endpoint deleted successfully to prevent charges.
